# Day 2: N-grams and Melodic Tendency
**Date:** Tuesday 23 June 2026

**Conceptual frame:** Pitch distributions tell us *what is present*.
Transition probabilities tell us *what tends to follow what*.
These are different theories of musical pattern.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 3 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 3 Quiz](../quizpages/day3_quiz.md)**
```


---
## Part 1: Setup


In [ ]:
import requests, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from music21 import converter, note, interval
sns.set_theme(style='whitegrid',font_scale=1.1)
plt.rcParams['figure.figsize']=(10,4)
print('OK')

In [ ]:
CORPUS_DIR=Path('beregovski_corpus');KERN_DIR=CORPUS_DIR/'kern'
if not(KERN_DIR.exists() and list(KERN_DIR.glob('*.krn'))):
    CORPUS_DIR.mkdir(exist_ok=True)
    r=requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp=CORPUS_DIR/'repo.zip';zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z:z.extractall(CORPUS_DIR)
    src=list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists():shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0],KERN_DIR);zp.unlink()
print(f'{len(list(KERN_DIR.glob("*.krn")))} kern files ready')

In [ ]:
def load_corpus(kern_dir=KERN_DIR, verbose=True):
    pc2d={7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records,sdict={},{}
    for i,f in enumerate(sorted(Path(kern_dir).glob('*.krn'))):
        if verbose and i%50==0: print(f'  {i+1}...')
        try:
            s=converter.parse(str(f));ns=[n for n in s.flat.notes if isinstance(n,note.Note)]
            pcs=[n.pitch.pitchClass for n in ns]
            records[f.stem]={'tune_id':f.stem,'n_notes':len(ns),
                'pitches':[n.nameWithOctave for n in ns],'pitch_classes':pcs,
                'scale_degrees':[pc2d.get(p,0) for p in pcs],
                'intervals':[interval.Interval(ns[j],ns[j+1]).semitones for j in range(len(ns)-1)]}
            sdict[f.stem]=s
        except: pass
    if verbose: print(f'Loaded {len(records)} tunes.')
    return pd.DataFrame(records.values()),sdict

def get_ngrams(seq,n): return list(zip(*[islice(seq,i,None) for i in range(n)]))
print('Helpers ready.')

In [ ]:
df,streams=load_corpus()
try:
    meta=pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df=df.merge(meta,on='tune_id',how='left')
    print(f'{len(df)} tunes loaded with metadata')
except Exception as e:
    print(f'Metadata unavailable ({e})')

---
## Part 2: Melodic Bigrams by Mode


In [ ]:
def mode_bigrams(df_,mode,col='scale_degrees'):
    subset=df_[df_['mode']==mode] if 'mode' in df_.columns else df_
    bg=[]
    for seq in subset[col]: bg.extend(get_ngrams([x for x in seq if x!=0],2))
    return Counter(bg)

modes=[m for m in ['freygish','raised_fourth','minor','major']
       if 'mode' in df.columns and m in df['mode'].values]
bigram_counters={m:mode_bigrams(df,m) for m in modes}

for mode,counter in bigram_counters.items():
    print(f'\n{mode.upper()} — top 8 scale-degree bigrams:')
    for bg,c in counter.most_common(8): print(f'  {bg[0]}→{bg[1]}: {c}')

In [ ]:
fig,axes=plt.subplots(1,len(modes),figsize=(14,4),sharey=True)
colors={'freygish':'steelblue','raised_fourth':'coral','minor':'seagreen','major':'mediumpurple'}
for ax,mode in zip(axes,modes):
    top=bigram_counters[mode].most_common(10)
    ax.barh([f'{a}→{b}' for (a,b),_ in top][::-1],[c for _,c in top][::-1],
            color=colors.get(mode,'gray'))
    ax.set_title(mode);ax.set_xlabel('Count')
fig.suptitle('Top 10 scale-degree bigrams by mode',fontsize=12)
plt.tight_layout();plt.show()

---
## Part 3: Bigram Similarity Between Modes


In [ ]:
def jaccard(c1,c2,n=50):
    s1=set(dict(c1.most_common(n)).keys())
    s2=set(dict(c2.most_common(n)).keys())
    return len(s1&s2)/len(s1|s2) if s1|s2 else 1.0

print('Bigram Jaccard similarity (top-50):')
for m1 in modes:
    row=' '.join(f'{jaccard(bigram_counters[m1],bigram_counters[m2]):6.3f}' for m2 in modes)
    print(f'  {m1:<15s}: {row}')
print('Columns:', modes)

---
## Part 4: Interval Bigrams
Scale-degree bigrams describe *what* follows what. Interval bigrams describe *how far*.


In [ ]:
ivl_bigrams={m:Counter() for m in modes}
for m in modes:
    subset=df[df['mode']==m] if 'mode' in df.columns else df
    for ivls in subset['intervals']:
        ivl_bigrams[m].update(get_ngrams(ivls,2))

print('Top interval bigrams (semitones) by mode:')
for mode in modes:
    top=ivl_bigrams[mode].most_common(5)
    print(f'  {mode}: {[(f"{a}→{b}",c) for (a,b),c in top]}')

---
## Day 3 Exercise: Bigram Argument

```{admonition} Exercise
Extract bigrams for two different modes. Find the single transition most overrepresented
in one mode relative to the other. Write 150–200 words: what does this transition tell us
about modal character, and is it something a performer would recognize — or only a corpus analyst?
```


In [ ]:
MODE_A='freygish';MODE_B='minor'  # <-- change

if MODE_A in bigram_counters and MODE_B in bigram_counters:
    ca,cb=bigram_counters[MODE_A],bigram_counters[MODE_B]
    ta,tb=sum(ca.values()) or 1,sum(cb.values()) or 1
    diffs={bg:(ca.get(bg,0)/ta)-(cb.get(bg,0)/tb) for bg in set(ca)|set(cb)}
    top=sorted(diffs.items(),key=lambda x:abs(x[1]),reverse=True)[:10]
    print(f'Most distinctive bigrams ({MODE_A} vs {MODE_B}):')
    for bg,d in top:
        print(f'  {bg[0]}→{bg[1]}: {d:+.4f}  ({"more in "+MODE_A if d>0 else "more in "+MODE_B})')

### Your argument

*(Write 150–200 words here)*


---
## Project Log — Entry 3

> *The most characteristic transitions in my subset are [X].*  
> *They differ from the full-corpus mode profile in [Y] ways.*  
> *Comparing to Parker bigrams, I notice...*

*(100–150 words)*
